In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pandas pyarrow lxml beautifulsoup4 pymupdf -q

In [ ]:
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

In [ ]:
!wget -q https://github.com/kermitt2/grobid/archive/0.8.2.zip
!unzip -qq 0.8.2.zip
%cd grobid-0.8.2
!./gradlew clean install --no-daemon -q

In [ ]:
import subprocess
log = open("/content/grobid.log", "w")
subprocess.Popen(["./gradlew", "run", "--no-daemon"], stdout=log, stderr=subprocess.STDOUT)

In [ ]:
import time, requests
for i in range(60):
    try:
        if requests.get("http://localhost:8070/api/isalive", timeout=5).text.strip() == "true":
            print("GROBID up"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print("not up — check /content/grobid.log")

In [ ]:
!pip install -q grobid-client-python
from grobid_client.grobid_client import GrobidClient

client = GrobidClient(grobid_server="http://localhost:8070", timeout=120)
client.process("processFulltextDocument",
               "/content/pdfs", output="/content/tei",
               n=8, consolidate_citations=True, tei_coordinates=False)

In [ ]:
import subprocess, os
GROBID_HOME = "/content/grobid-0.8.2"
assert os.path.exists(GROBID_HOME + "/gradlew"), "build dir missing — rerun the wget/unzip/build cells"
log = open("/content/grobid.log", "w")
subprocess.Popen(["./gradlew", "run", "--no-daemon"],
                 cwd=GROBID_HOME, stdout=log, stderr=subprocess.STDOUT,
                 env={**os.environ, "JAVA_HOME": "/usr/lib/jvm/java-11-openjdk-amd64"})

In [ ]:
# import subprocess, os, requests, time
# GROBID_HOME = "/content/grobid-0.8.2"
# log = open("/content/grobid.log", "w")
# subprocess.Popen(["./gradlew", "run", "--no-daemon"], cwd=GROBID_HOME,
#                  stdout=log, stderr=subprocess.STDOUT,
#                  env={**os.environ, "JAVA_HOME": "/usr/lib/jvm/java-11-openjdk-amd64"})

# BASE = "http://127.0.0.1:8070"
# for i in range(40):
#     try:
#         r = requests.get(f"{BASE}/api/isalive", timeout=5)
#         print(i, r.status_code, repr(r.text))
#         if r.text.strip().lower() == "true":
#             print("UP ->", BASE); break
#     except Exception as e:
#         print(i, type(e).__name__, str(e)[:120])
#     time.sleep(15)

In [ ]:
#!/usr/bin/env python3
"""
reparse_offline_v2.py  —  Google Colab
======================================
Re-découpe les fichiers BRUTS déjà présents avec la détection d'en-têtes AMÉLIORÉE
(tick_sections.py). AUCUN TÉLÉCHARGEMENT.

v2 — GESTION DU PROBLÈME GOOGLE DRIVE
  Drive monté ne supporte pas de parcourir/lire/écrire des dizaines de milliers de
  petits fichiers : `OSError [Errno 5] Input/output error`, timeouts, dossiers qui
  apparaissent vides. Ce script contourne cela :

   1. COPIE LOCALE PAR LOTS  — copie un lot de fichiers bruts de Drive vers le
      disque local de Colab, travaille en local (10x plus rapide, fiable), puis
      renvoie les JSON améliorés vers Drive. Jamais plus de BATCH fichiers à la
      fois, donc jamais de gros listing Drive.
   2. LISTING ROBUSTE        — le listing de raw/ est mis en cache dans un fichier
      local ; en cas d'échec du listing Drive, on réessaie avec backoff.
   3. TOLÉRANCE AUX ERREURS  — toute erreur d'E/S sur un fichier est capturée,
      comptée, et le traitement continue.
   4. REPRISE                — un journal local note chaque oa_id traité ; relancer
      reprend où l'exécution s'est arrêtée.
   5. SYNCHRO INCRÉMENTALE   — les JSON améliorés sont renvoyés vers Drive après
      chaque lot, pas à la fin (rien n'est perdu si la session tombe).

RÈGLE DE SÉCURITÉ : un JSON n'est réécrit que si le nouveau découpage apporte des
sections cibles que l'ancien n'avait pas. Jamais de régression.

SETUP
  !pip install pandas pyarrow lxml beautifulsoup4 pymupdf -q
"""

import json, os, re, shutil, sys, time
from collections import Counter
from pathlib import Path
import pandas as pd
from lxml import etree

# --------------------------------------------------------------------------- #
# module partagé de détection d'en-têtes
# --------------------------------------------------------------------------- #
sys.path.insert(0, "/content/drive/MyDrive/Colab Notebooks")
sys.path.insert(0, "/content")
from tick_sections import (classify_line, classify_heading, parse_plaintext,
                           has_targets, TARGET_SECTIONS)

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
DRIVE_FT  = "/content/drive/MyDrive/COGNOSX/ticks/fulltext2"
DRIVE_RAW = DRIVE_FT + "/raw"
DRIVE_SEC = DRIVE_FT + "/sections"
MANIFEST  = DRIVE_FT + "/fulltext_manifest.parquet"
REPORT    = DRIVE_FT + "/reparse_report.csv"

LOCAL     = "/content/_reparse"          # espace de travail local
LOCAL_RAW = LOCAL + "/raw"
LOCAL_SEC = LOCAL + "/sections"
PROGRESS  = LOCAL + "/progress.jsonl"    # journal de reprise (local)
LISTING   = LOCAL + "/raw_listing.txt"   # cache du listing Drive

BATCH          = 500          # fichiers copiés/traités par lot
GROBID_URL     = "http://localhost:8070"
MIN_BODY_CHARS = 500
DRY_RUN        = False
MAX_BATCHES    = 0            # 0 = tout ; sinon limite (pour un essai)

for d in (LOCAL, LOCAL_RAW, LOCAL_SEC):
    Path(d).mkdir(parents=True, exist_ok=True)


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def _txt(el):
    return re.sub(r"\s+", " ", "".join(el.itertext())).strip() if el is not None else ""


# =========================================================================== #
# 1. LISTING ROBUSTE de raw/ (avec cache local + backoff)                      #
# =========================================================================== #
def list_raw_files():
    """Liste raw/ une seule fois, met en cache. Réessaie si Drive échoue."""
    if Path(LISTING).exists():
        names = [l.strip() for l in open(LISTING) if l.strip()]
        print(f"[listing] {len(names):,} fichiers (cache local)", file=sys.stderr)
        return names
    for attempt in range(5):
        try:
            names = sorted(p.name for p in Path(DRIVE_RAW).iterdir() if p.is_file())
            Path(LISTING).write_text("\n".join(names))
            print(f"[listing] {len(names):,} fichiers listés depuis Drive", file=sys.stderr)
            return names
        except OSError as e:
            wait = 5 * (attempt + 1)
            print(f"[listing] échec Drive ({e}); nouvelle tentative dans {wait}s",
                  file=sys.stderr)
            time.sleep(wait)
    raise SystemExit("[!] impossible de lister raw/ — remontez Drive "
                     "(drive.mount(force_remount=True)) puis relancez")


def load_progress():
    done = set()
    if Path(PROGRESS).exists():
        for line in open(PROGRESS):
            try:
                done.add(json.loads(line)["oa_id"])
            except Exception:
                pass
    if done:
        print(f"[reprise] {len(done):,} fichiers déjà traités", file=sys.stderr)
    return done


# =========================================================================== #
# 2. parseurs (inchangés, via le module partagé)                              #
# =========================================================================== #
SEC_TYPE_MAP = {"intro": "introduction", "introduction": "introduction",
    "methods": "methods", "materials|methods": "methods", "results": "results",
    "discussion": "discussion", "conclusion": "conclusion",
    "data-availability": "data_availability", "ethics-statement": "ethics",
    "supplementary-material": "supplementary", "software-availability": "software"}

def parse_jats(xml_text):
    try:
        xml_text = re.sub(r"<!DOCTYPE[^>]*>", "", xml_text)
        root = etree.fromstring(xml_text.encode("utf-8"),
                                parser=etree.XMLParser(recover=True, resolve_entities=False))
    except Exception:
        return None
    out = {"sections": [], "refs": [],
           "language": root.get("{http://www.w3.org/XML/1998/namespace}lang") or ""}
    for p, lab in [(".//article-title", "title"), (".//abstract", "abstract")]:
        el = root.xpath(p)
        if el: out["sections"].append({"type": lab, "title": lab, "text": _txt(el[0])})
    for sec in root.xpath(".//body//sec"):
        stype = (sec.get("sec-type") or "").lower()
        title = _txt(sec.find("title"))
        label = (SEC_TYPE_MAP.get(stype) or classify_line(title)[0]
                 or classify_heading(title) or (stype or "section"))
        text = " ".join(x for x in (_txt(p) for p in sec.findall("p")) if x)
        if text: out["sections"].append({"type": label, "title": title, "text": text})
    for sec in (root.xpath(".//back//sec") + root.xpath(".//back//ack")
                + root.xpath(".//back//notes")):
        stype = (sec.get("sec-type") or "").lower()
        title = _txt(sec.find("title"))
        label = (SEC_TYPE_MAP.get(stype) or classify_line(title)[0]
                 or classify_heading(title) or "back")
        text = _txt(sec)
        if text and len(text) > 5:
            out["sections"].append({"type": label, "title": title, "text": text})
    if not has_targets(out):
        body_txt = "\n".join(s["text"] for s in out["sections"])
        alt = parse_plaintext(body_txt, MIN_BODY_CHARS)
        if alt and has_targets(alt):
            for s in alt["sections"]:
                if s["type"] in TARGET_SECTIONS:
                    out["sections"].append(s)
    for ref in root.xpath(".//ref-list//ref"):
        doi = ""
        for pid in ref.xpath(".//pub-id"):
            if (pid.get("pub-id-type") or "") == "doi": doi = _txt(pid)
        out["refs"].append({"text": _txt(ref), "doi": doi})
    body = sum(len(s["text"]) for s in out["sections"]
               if s["type"] not in ("title", "abstract"))
    return out if body >= MIN_BODY_CHARS else None

def parse_tei(tei_text):
    try:
        tei_text = re.sub(r"<\?xml[^>]*\?>", "", tei_text)
        root = etree.fromstring(tei_text.encode("utf-8"), parser=etree.XMLParser(recover=True))
    except Exception:
        return None
    ns = {"t": "http://www.tei-c.org/ns/1.0"}
    out = {"sections": [], "refs": [], "language": ""}
    t = root.xpath("//t:titleStmt/t:title", namespaces=ns)
    if t: out["sections"].append({"type": "title", "title": "title", "text": _txt(t[0])})
    ab = root.xpath("//t:profileDesc//t:abstract", namespaces=ns)
    if ab: out["sections"].append({"type": "abstract", "title": "abstract", "text": _txt(ab[0])})
    for div in root.xpath("//t:body/t:div", namespaces=ns):
        head = div.xpath("t:head", namespaces=ns)
        title = _txt(head[0]) if head else ""
        text = " ".join(_txt(p) for p in div.xpath("t:p", namespaces=ns))
        if text:
            lab = classify_line(title)[0] or classify_heading(title) or "section"
            out["sections"].append({"type": lab, "title": title, "text": text})
    for bib in root.xpath("//t:listBibl/t:biblStruct", namespaces=ns):
        out["refs"].append({"text": _txt(bib), "doi": ""})
    body = sum(len(s["text"]) for s in out["sections"]
               if s["type"] not in ("title", "abstract"))
    return out if body >= MIN_BODY_CHARS else None

def pdf_text(b):
    try:
        import fitz
        fitz.TOOLS.mupdf_display_errors(False)
        doc = fitz.open(stream=b, filetype="pdf")
        return "\n".join(p.get_text() for p in doc)
    except Exception:
        return ""

def html_text(b):
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(b, "html.parser")
        for t in soup(["script", "style", "nav", "header", "footer"]): t.decompose()
        art = (soup.find("article")
               or soup.find("div", {"class": re.compile("article|fulltext|content")})
               or soup)
        return re.sub(r"\n{3,}", "\n\n", art.get_text("\n"))
    except Exception:
        return ""

def grobid(b):
    if not GROBID_URL: return None
    try:
        import requests
        r = requests.post(f"{GROBID_URL}/api/processFulltextDocument",
                          files={"input": ("a.pdf", b, "application/pdf")}, timeout=180)
        return r.text if r.status_code == 200 else None
    except Exception:
        return None

def reparse_raw(path):
    try:
        b = path.read_bytes()
    except Exception:
        return None, "unreadable"
    if not b:
        return None, "empty"
    is_pdf = b[:5].startswith(b"%PDF")
    head = b[:2000].lower()
    is_html = (b"<html" in head or b"<!doctype html" in head)
    is_xml = b.lstrip()[:6].startswith(b"<?xml") or b"<article" in head
    if is_pdf:
        tei = grobid(b)
        if tei:
            p = parse_tei(tei)
            if p: return p, "grobid"
        txt = pdf_text(b)
        if txt:
            p = parse_plaintext(txt, MIN_BODY_CHARS)
            if p: return p, "pymupdf_regex_v3"
        return None, "pdf_unparsed"
    if is_xml:
        p = parse_jats(b.decode("utf-8", "ignore"))
        return (p, "jats_v3") if p else (None, "xml_unparsed")
    if is_html:
        txt = html_text(b)
        if txt:
            p = parse_plaintext(txt, MIN_BODY_CHARS)
            if p: return p, "html_regex_v3"
        return None, "html_unparsed"
    txt = b.decode("utf-8", "ignore")
    p = parse_plaintext(txt, MIN_BODY_CHARS)
    return (p, "text_regex_v3") if p else (None, "unknown_format")


# =========================================================================== #
# 3. E/S Drive tolérantes aux erreurs                                         #
# =========================================================================== #
def safe_copy(src, dst, tries=3):
    for a in range(tries):
        try:
            shutil.copy(src, dst); return True
        except OSError:
            time.sleep(2 * (a + 1))
    return False

def safe_read_json(path, tries=3):
    for a in range(tries):
        try:
            return json.loads(Path(path).read_text(encoding="utf-8"))
        except FileNotFoundError:
            return None
        except OSError:
            time.sleep(1 * (a + 1))
        except Exception:
            return None
    return None


# =========================================================================== #
# 4. traitement d'un lot : Drive -> local -> traitement -> Drive              #
# =========================================================================== #
def process_batch(names, stats, rows, fh_prog):
    # (a) copier les bruts de ce lot vers le local
    copied = []
    for nm in names:
        dst = Path(LOCAL_RAW) / nm
        if dst.exists() or safe_copy(Path(DRIVE_RAW) / nm, dst):
            copied.append(dst)
        else:
            stats["copy_failed"] += 1
    # (b) copier les JSON existants correspondants (pour comparer)
    for p in copied:
        oid = p.stem
        d = Path(LOCAL_SEC) / f"{oid}.json"
        if not d.exists():
            safe_copy(Path(DRIVE_SEC) / f"{oid}.json", d)   # peut ne pas exister

    # (c) traiter en local
    improved = []
    for p in copied:
        oid = p.stem
        old = safe_read_json(Path(LOCAL_SEC) / f"{oid}.json")
        old_tgt = bool(old) and (
            bool(old.get("has_targets"))
            or bool({s.get("type") for s in old.get("sections", [])} & TARGET_SECTIONS))
        try:
            new, method = reparse_raw(p)
        except Exception:
            new, method = None, "exception"
        new_tgt = bool(new) and has_targets(new)
        gained = new_tgt and not old_tgt
        stats[method] += 1
        if gained: stats["_GAINED"] += 1
        elif new and not old: stats["_NEW"] += 1

        rows.append({"oa_id": oid, "ext": p.suffix, "method": method,
                     "old_has_targets": old_tgt, "new_has_targets": new_tgt,
                     "gained": gained,
                     "old_sections": len(old.get("sections", [])) if old else 0,
                     "new_sections": len(new["sections"]) if new else 0})
        fh_prog.write(json.dumps({"oa_id": oid, "gained": gained}) + "\n")

        if new and (gained or (not old and new)) and not DRY_RUN:
            doc = {"oa_id": oid, "oa_doi": (old or {}).get("oa_doi", ""),
                   "oa_pmcid": (old or {}).get("oa_pmcid", ""),
                   "source": ((old or {}).get("source", "") or method) + "+reparse_v3",
                   "language": new.get("language", "") or (old or {}).get("language", ""),
                   "sections": new["sections"], "refs": new.get("refs", []),
                   "has_targets": True}
            lp = Path(LOCAL_SEC) / f"{oid}.json"
            lp.write_text(json.dumps(doc, ensure_ascii=False), encoding="utf-8")
            improved.append(lp)

    fh_prog.flush(); os.fsync(fh_prog.fileno())

    # (d) renvoyer vers Drive UNIQUEMENT les JSON améliorés
    if not DRY_RUN:
        for lp in improved:
            if not safe_copy(lp, Path(DRIVE_SEC) / lp.name):
                stats["sync_failed"] += 1

    # (e) libérer l'espace local de ce lot
    for p in copied:
        try: p.unlink()
        except Exception: pass
    return len(improved)


# =========================================================================== #
if __name__ == "__main__":
    names = list_raw_files()
    done = load_progress()
    todo = [n for n in names if Path(n).stem not in done]
    print(f"[in] {len(names):,} bruts | {len(todo):,} à traiter", file=sys.stderr)
    if not todo:
        print("[ok] tout est déjà traité", file=sys.stderr); raise SystemExit

    stats, rows = Counter(), []
    fh_prog = open(PROGRESS, "a")
    t0 = time.time()
    batches = [todo[i:i + BATCH] for i in range(0, len(todo), BATCH)]
    if MAX_BATCHES: batches = batches[:MAX_BATCHES]

    try:
        for bi, batch in enumerate(batches, 1):
            n_imp = process_batch(batch, stats, rows, fh_prog)
            el = (time.time() - t0) / 60
            print(f"[lot {bi:,}/{len(batches):,}] traités {len(rows):,} | "
                  f"gagnés {stats['_GAINED']:,} | améliorés ce lot {n_imp} | {el:.1f} min",
                  file=sys.stderr)
    except KeyboardInterrupt:
        print("\n[interrompu] progression sauvegardée — relancez pour reprendre",
              file=sys.stderr)
    finally:
        fh_prog.close()

    rep = pd.DataFrame(rows)
    if len(rep):
        local_report = LOCAL + "/reparse_report.csv"
        rep.to_csv(local_report, index=False)
        safe_copy(local_report, REPORT)

        print("\n" + "=" * 62)
        print("RÉSULTAT DU REPARSING (aucun téléchargement)")
        print("=" * 62)
        print(f"  fichiers traités                 : {len(rep):,}")
        print(f"  avaient déjà des sections cibles : {int(rep['old_has_targets'].sum()):,}")
        print(f"  en ont maintenant                : {int(rep['new_has_targets'].sum()):,}")
        print(f"  >>> GAIN NET                     : {int(rep['gained'].sum()):,}")
        print("\n  par méthode :")
        for k, v in rep["method"].value_counts().items():
            print(f"    {k:<24}{v:>8,}")
        if stats["copy_failed"] or stats["sync_failed"]:
            print(f"\n  [!] erreurs Drive — copie: {stats['copy_failed']:,}, "
                  f"synchro: {stats['sync_failed']:,} (relancez pour les reprendre)")

        # manifeste
        if Path(MANIFEST).exists() and not DRY_RUN:
            try:
                man = pd.read_parquet(MANIFEST)
                gained_ids = set(rep.loc[rep["gained"], "oa_id"])
                if gained_ids:
                    m = man["oa_id"].isin(gained_ids)
                    man.loc[m, ["has_targets", "ok"]] = True
                    man.to_parquet(MANIFEST, index=False)
                    print(f"\n[manifest] {int(m.sum()):,} lignes -> has_targets=True",
                          file=sys.stderr)
            except OSError as e:
                print(f"[manifest] écriture Drive échouée ({e}) — relancez plus tard",
                      file=sys.stderr)
    if DRY_RUN:
        print("\n[DRY RUN] rien n'a été réécrit", file=sys.stderr)